In [ ]:
#Gustavo Fraiz E Gustavo Albiero

In [ ]:
!pip install transformers sentence-transformers bert-score pandas

In [ ]:
frases_originais = []


with open("/content/frases.txt", "r") as f:
    for linha in f:
        frases_originais.append(linha.strip())

print(f"Carregadas {len(frases_originais)} frases originais:")
print(frases_originais)

In [ ]:

import random
from transformers import pipeline


filler = pipeline(
    'fill-mask',
    model='neuralmind/bert-base-portuguese-cased',
    top_k=5,
    device=0
)

frases_parafraseadas = []
for frase in frases_originais:
    words = frase.split()

    if len(words) < 4:
        frases_parafraseadas.append(frase)
        print(f"Original (curta): {frase}")
        print(f"Gerada (mantida): {frase}")
        print("-" * 30)
        continue

    eligible_indices = [i for i, word in enumerate(words) if len(word) >= 4]

    random.shuffle(eligible_indices)

    num_to_swap_target = max(1, int(len(words) * 0.30))

    indices_to_swap = []

    for idx in eligible_indices:
        is_too_close = False
        for chosen_idx in indices_to_swap:
            if abs(idx - chosen_idx) <= 3:
                is_too_close = True
                break

        if not is_too_close:
            indices_to_swap.append(idx)

        if len(indices_to_swap) >= num_to_swap_target:
            break

    new_words = list(words)

    for idx in indices_to_swap:
        original_word = new_words[idx]

        temp_words = list(new_words)
        temp_words[idx] = filler.tokenizer.mask_token
        masked_string = " ".join(temp_words)

        predictions = filler(masked_string)

        found_swap = False
        for pred in predictions:
            if pred['token_str'].lower() != original_word.lower():
                new_words[idx] = pred['token_str']
                found_swap = True
                break

        if not found_swap:
            new_words[idx] = original_word

    paraphrase = " ".join(new_words)
    frases_parafraseadas.append(paraphrase)

    print(f"Original: {frase}")
    print(f"Gerada:   {paraphrase}")
    print("-" * 30)

print("Geração de paráfrases concluída.")

In [ ]:
from bert_score import score

(P, R, F1) = score(
    frases_parafraseadas,
    frases_originais,
    lang="pt",
    verbose=True
)


scores_bertscore_f1 = F1.tolist()

print("\nScores F1 do BERTScore:")
print(scores_bertscore_f1)

In [ ]:
from sentence_transformers import SentenceTransformer, util


sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')


embeddings_originais = sbert_model.encode(frases_originais, convert_to_tensor=True)
embeddings_parafraseadas = sbert_model.encode(frases_parafraseadas, convert_to_tensor=True)


scores_sbert = []
for i in range(len(frases_originais)):
    score = util.pytorch_cos_sim(embeddings_originais[i], embeddings_parafraseadas[i])
    scores_sbert.append(score.item())

print("\nScores de Similaridade de Cosseno (SBERT):")
print(scores_sbert)

In [ ]:
import pandas as pd

data = {
    "Frase Original": frases_originais,
    "Frase Parafraseada": frases_parafraseadas,
    "BERTScore F1": scores_bertscore_f1,
    "SBERT Similaridade": scores_sbert
}

df_comparativo = pd.DataFrame(data)

pd.set_option('display.max_colwidth', None)

print("Tabela Comparativa Final")
display(df_comparativo)